In [ ]:
FIGURE = 'Fig.S18'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
layout_checks = {}
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap
from scipy.stats import t as student_t
BASE = Path(asset_path('asset_a2eff2840706'))
OUT_DIR = BASE / '00partial_correlations'
NC_FILE = INPUT_DIR / 'partial_correlations.nc'
MASK_FILE = Path(asset_path('asset_5e3c32038bfb'))
SEASONS = ['MAM']
SCALES = [1, 3, 6, 12, 24]

def load_mask(lat: np.ndarray, lon: np.ndarray) -> np.ndarray | None:
    if not MASK_FILE.exists():
        return None
    with rasterio.open(MASK_FILE) as src:
        arr = src.read(1)
        nodata = src.nodata
        mask = np.isfinite(arr)
        if nodata is not None:
            mask &= arr != nodata
        mask &= arr > 0
        if mask.shape != (len(lat), len(lon)):
            print(f'Mask shape {mask.shape} does not match data {(len(lat), len(lon))}; skip external mask.')
            return None
        if src.transform.e < 0 and lat[0] < lat[-1]:
            mask = np.flipud(mask)
        return mask

def discrete_r_colormap():
    bounds = np.array([-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    colors = ['#08306B', '#2171B5', '#6BAED6', '#BDD7E7', '#EFF3FF', '#FFF5EB', '#FDD0A2', '#FDAE6B', '#E6550D', '#7F2704']
    cmap = ListedColormap(colors)
    cmap.set_bad((1, 1, 1, 0))
    norm = BoundaryNorm(bounds, cmap.N)
    return (cmap, norm, bounds)

def add_geo_style(ax, row: int, col: int):
    ax.set_global()
    ax.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#F4F1E6', edgecolor='none', zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale('110m'), linewidth=0.35, edgecolor='0.25', zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale('110m'), linewidth=0.15, edgecolor='0.55', zorder=3)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.25, color='0.82', alpha=0.8, linestyle='-')
    gl.top_labels = False
    gl.right_labels = col == len(SCALES) - 1
    gl.left_labels = col == 0
    gl.bottom_labels = row == len(SEASONS) - 1
    ax._layout_gridliner = gl
    gl.xlabel_style = {'size': 7, 'fontname': 'Arial'}
    gl.ylabel_style = {'size': 7, 'fontname': 'Arial'}
    ticks = [-120, -60, 0, 60, 120]
    if col == 0:
        ticks.insert(0, -180)
    if col == len(SCALES) - 1:
        ticks.append(180)
    gl.xlocator = mpl.ticker.FixedLocator(ticks)
    gl.ylocator = mpl.ticker.FixedLocator([-60, -30, 0, 30, 60])

def downsample_points(lon2d: np.ndarray, lat2d: np.ndarray, sig: np.ndarray, step: int=5):
    sig_s = sig[::step, ::step]
    y, x = np.where(sig_s)
    return (lon2d[::step, ::step][y, x], lat2d[::step, ::step][y, x])

def class_percentages(data: np.ndarray, bounds: np.ndarray) -> np.ndarray:
    valid = np.isfinite(data)
    if not valid.any():
        return np.full(len(bounds) - 1, np.nan, dtype=np.float32)
    values = np.clip(data[valid], bounds[0], bounds[-1])
    counts, _ = np.histogram(values, bins=bounds)
    return counts.astype(np.float32) / counts.sum() * 100.0

def p_from_r(r: np.ndarray, n: np.ndarray, control_vars: int=0) -> np.ndarray:
    df = n - 2 - control_vars
    p = np.full_like(r, np.nan, dtype=np.float32)
    ok = np.isfinite(r) & np.isfinite(n) & (df > 0) & (np.abs(r) < 1)
    tval = np.abs(r[ok]) * np.sqrt(df[ok] / np.maximum(1e-12, 1.0 - r[ok] * r[ok]))
    p[ok] = (2.0 * student_t.sf(tval, df[ok])).astype(np.float32)
    p[np.isfinite(r) & (np.abs(r) >= 1) & (df > 0)] = 0.0
    return p

def fdr_bh(p: np.ndarray) -> np.ndarray:
    flat = p.ravel()
    q = np.full_like(flat, np.nan, dtype=np.float32)
    valid = np.isfinite(flat)
    pv = flat[valid].astype(np.float64)
    m = pv.size
    if m == 0:
        return q.reshape(p.shape)
    order = np.argsort(pv)
    ranked = pv[order]
    raw = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(raw[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    tmp = np.empty_like(adj)
    tmp[order] = adj
    q[valid] = tmp.astype(np.float32)
    return q.reshape(p.shape)

def add_distribution_inset(ax, data: np.ndarray, bounds: np.ndarray, colors: list[str], sig_pct: float | None):
    pct = class_percentages(data, bounds)
    iax = ax.inset_axes([0.015, 0.025, 0.36, 0.3])
    iax.set_facecolor((1, 1, 1, 0))
    for name, spine in iax.spines.items():
        spine.set_visible(name == 'bottom')
        spine.set_linewidth(0.35)
    x = np.arange(len(pct))
    safe_pct = np.nan_to_num(pct, nan=0.0)
    ymax = max(20.0, float(np.nanmax(safe_pct)) * 1.22)
    ymax = min(100.0, np.ceil(ymax / 10.0) * 10.0)
    iax.bar(x, safe_pct, color=colors, width=0.82, edgecolor='none')
    iax.set_ylim(0, ymax)
    iax.set_xlim(-0.55, len(pct) - 0.45)
    iax.set_xticks([])
    iax.set_yticks([])
    iax.tick_params(length=0)
    for xi, yi in zip(x, safe_pct):
        if yi >= 8.0:
            iax.text(xi, yi + ymax * 0.025, f'{yi:.0f}', ha='center', va='bottom', fontsize=3.8, fontname='Arial')
    if sig_pct is not None and np.isfinite(sig_pct):
        iax.text(0.02, 0.98, f'Sig {sig_pct:.1f}%', transform=iax.transAxes, ha='left', va='top', fontsize=4.6, fontname='Arial', color='black')
    iax.text(0.98, 0.02, '%', transform=iax.transAxes, ha='right', va='bottom', fontsize=4.2, fontname='Arial', color='0.25')

def plot_grid(ds, var, qvar, title, out_png, cbar_label, mask):
    lat = ds['lat'].values
    lon = ds['lon'].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    cmap, norm, bounds = discrete_r_colormap()
    plt.rcParams.update({'font.family': 'Arial', 'axes.linewidth': 0.5, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    fig = plt.figure(figsize=(15.6, 6.6), dpi=600)
    plot_height = fig.get_figheight() * (0.93 - 0.11)
    map_height = plot_height / (len(SEASONS) + 0.08 * (len(SEASONS) - 1))
    gap = 0.08 * map_height
    map_width = 2 * map_height
    plot_width = len(SCALES) * map_width + (len(SCALES) - 1) * gap
    left = (fig.get_figwidth() - plot_width) / (2 * fig.get_figwidth())
    gs = fig.add_gridspec(nrows=len(SEASONS), ncols=len(SCALES), left=left, right=1 - left, top=0.93, bottom=0.11, wspace=gap / map_width, hspace=gap / map_height)
    proj = ccrs.PlateCarree()
    im = None
    for ri, season in enumerate(SEASONS):
        for ci, scale in enumerate(SCALES):
            ax = fig.add_subplot(gs[ri, ci], projection=proj)
            add_geo_style(ax, ri, ci)
            data = ds[var].sel(season=season, scale=scale).values.astype(np.float32)
            if mask is not None:
                data = np.where(mask, data, np.nan)
            im = ax.pcolormesh(lon, lat, data, cmap=cmap, norm=norm, shading='auto', transform=proj, zorder=2)
            sig_pct = None
            if qvar is not None:
                q = ds[qvar].sel(season=season, scale=scale).values
                sig = np.isfinite(data) & np.isfinite(q) & (q < 0.05)
                if mask is not None:
                    sig &= mask
                valid_for_sig = np.isfinite(data) & np.isfinite(q)
                if mask is not None:
                    valid_for_sig &= mask
                if valid_for_sig.any():
                    sig_pct = float(sig.sum() / valid_for_sig.sum() * 100.0)
                px, py = downsample_points(lon2d, lat2d, sig, step=5)
                ax.scatter(px, py, s=0.55, c='black', marker='.', linewidths=0, alpha=0.85, transform=proj, zorder=4)
            elif var == 'spearman_r_SPEI_SSMIrz':
                n = ds['n_valid'].sel(season=season, scale=scale).values
                p = p_from_r(data, n, control_vars=0)
                q = fdr_bh(p)
                valid_for_sig = np.isfinite(data) & np.isfinite(q)
                if mask is not None:
                    valid_for_sig &= mask
                sig = valid_for_sig & (q < 0.05)
                if valid_for_sig.any():
                    sig_pct = float(sig.sum() / valid_for_sig.sum() * 100.0)
            add_distribution_inset(ax, data, bounds, cmap.colors, sig_pct)
            ax.text(0.015, 0.965, f'{season}  {scale}m', transform=ax.transAxes, ha='left', va='top', fontsize=8.5, fontname='Times New Roman', bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=1.4), zorder=5)
    fig.suptitle(title, fontsize=13, fontname='Arial', y=0.985)
    cax = fig.add_axes([0.16, 0.045, 0.68, 0.02])
    cb = fig.colorbar(im, cax=cax, orientation='horizontal', boundaries=bounds, ticks=bounds, spacing='proportional', drawedges=True)
    cb.ax.tick_params(labelsize=7, length=2.5, width=0.5)
    cb.set_label(cbar_label, fontsize=9, fontname='Arial')
    fig.canvas.draw()
    maps = fig.axes[:len(SEASONS) * len(SCALES)]
    boxes = [axis.get_window_extent() for axis in maps]
    bottom_axes = maps[-len(SCALES):]
    bottom_boxes = boxes[-len(SCALES):]
    label_boxes = [artist.get_window_extent(fig.canvas.get_renderer()) for axis in bottom_axes for artist in axis._layout_gridliner.bottom_label_artists if artist.get_visible()]
    label_y = float(np.median([(box.y0 + box.y1) / 2 for box in label_boxes]))
    shared_longitudes = []
    for left_box, right_box in zip(bottom_boxes[:-1], bottom_boxes[1:]):
        x = (left_box.x1 + right_box.x0) / 2
        fx, fy = fig.transFigure.inverted().transform((x, label_y))
        fig.text(fx, fy, '180°', fontsize=7, fontname='Arial', ha='center', va='center')
        shared_longitudes.append([float(fx), float(fy)])
    horizontal = [boxes[r * len(SCALES) + c + 1].x0 - boxes[r * len(SCALES) + c].x1 for r in range(len(SEASONS)) for c in range(len(SCALES) - 1)]
    vertical = [boxes[r * len(SCALES) + c].y0 - boxes[(r + 1) * len(SCALES) + c].y1 for r in range(len(SEASONS) - 1) for c in range(len(SCALES))]
    assert max(horizontal + vertical) - min(horizontal + vertical) < 0.1
    for axis in maps:
        assert len(axis.child_axes) == 1
        assert axis.child_axes[0].patch.get_facecolor()[3] == 0
        assert [name for name, spine in axis.child_axes[0].spines.items() if spine.get_visible()] == ['bottom']
    layout_checks[var] = {'horizontal_gaps_px': horizontal, 'vertical_gaps_px': vertical, 'inset_bounds_axes_coordinates': [0.015, 0.025, 0.36, 0.3], 'inset_background_alpha': 0, 'dpi': 600, 'visible_inset_spines': ['bottom'], 'shared_180_degree_labels': shared_longitudes}
    fig.savefig(out_png, dpi=600, bbox_inches='tight')
    fig.savefig(out_png.with_suffix('.pdf'), dpi=600, bbox_inches='tight')
    plt.close(fig)

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    ds = xr.open_dataset(NC_FILE, engine='h5netcdf')
    mask = load_mask(ds['lat'].values, ds['lon'].values)
    jobs = [('partial_r_SIF_SSMIrz_ctrl_SPEI', 'partial_q_spatial_SIF_SSMIrz_ctrl_SPEI', 'Fixed-scale partial correlation: SIF-SSMIrz | SPEI', OUT_DIR / 'fixed_scale_partial_correlations_SIF-SSMIrzcontrollingSPEI_class_area_shares_1_3_6_12_24_600dpi.png', 'Spearman partial r classes')]
    for job in jobs:
        print('plot', job[3])
        plot_grid(ds, *job, mask=mask)
    ds.close()
if __name__ == '__main__':
    main()
